# Class 1 — Framing the Problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MostafaElFar96/QaF_Data_Analytics/blob/main/notebooks/Class_1_Framing.ipynb)

Module 4 · Sanad Distribution case (fictional data). Run the cells top to bottom. Works in **Google Colab** (nothing to install) and in **VS Code** (Python + Jupyter extension).

Class 1 has no database work. These cells reproduce the numbers used in the videos so you can change the assumptions and see what happens.

## Video 1 — One vehicle, one day: cost per drop from two starting points

In [ ]:
import pandas as pd

def cost_per_drop(one_way_km, round_trip_hours, workday_hours=8, minutes_per_drop=15,
                  vehicle_cost_day=3500, in_area_km=40, fuel_per_km=6):
    delivery_hours = workday_hours - round_trip_hours
    drops = int(delivery_hours * 60 // minutes_per_drop)
    fuel = (2 * one_way_km + in_area_km) * fuel_per_km
    return dict(drops=drops, fuel_egp=fuel, cost_per_drop=round((vehicle_cost_day + fuel) / drops, 1))

obour = cost_per_drop(one_way_km=35, round_trip_hours=2)
new_cairo = cost_per_drop(one_way_km=5, round_trip_hours=20/60)
pd.DataFrame({"From Obour (today)": obour, "From a site in New Cairo": new_cairo})

## Video 1 — Scaling it up

In [ ]:
drops_per_day, work_days = 400, 26
saving_per_drop = round(obour["cost_per_drop"]) - round(new_cairo["cost_per_drop"])   # 173 - 127
monthly_saving = drops_per_day * saving_per_drop * work_days
print(f"Saving per drop: EGP {saving_per_drop:.0f}  ->  about EGP {monthly_saving:,.0f} per month")

## Video 3 — The KPI table: main KPIs, sub-KPIs, current value, target, owner

Current values marked `None` are the question marks we measure from the data in Class 2.

In [ ]:
kpis = pd.DataFrame([
    ("Delivery cost % of order value", "main", None, "<= 4%", "Finance"),
    ("Next-day delivery %",            "main", None, "95%", "Customer Experience"),
    ("Cost per drop",                  "sub",  f"~EGP {obour['cost_per_drop']:.0f}", "<= EGP 130", "Operations"),
    ("Drops / vehicle / day",          "sub",  obour["drops"], 30, "Operations"),
    ("Drive time to farthest area",    "sub",  None, "<= 45 min", "Operations"),
    ("Same-day for key accounts",      "sub",  None, "100%", "Sales"),
], columns=["KPI", "level", "current", "target", "owner"])
kpis

In [ ]:
# The KPI tree as a simple nested structure
kpi_tree = {
    "Delivery cost % of order value": {"Cost per drop": ["Drops / vehicle / day", "Road time from warehouse", "Failed-delivery rate"]},
    "Next-day delivery %": ["Drive time to farthest area", "Same-day for key accounts"],
}
limits = {"Rent": "<= EGP 450K / month", "Payback": "<= 24 months"}
import json; print(json.dumps(kpi_tree, indent=2)); print("Limits (never exceeded):", limits)